# LightGBM (Default Params) — F1 Race Position Prediction

Trains a `LGBMRegressor` with default hyperparameters and evaluates it with **temporal cross-validation** — an expanding-window walk-forward split over seasons, so every fold is only ever validated on a season that comes *after* the seasons it was trained on. This mirrors the real deployment setting (predict an upcoming season using only past seasons) and avoids the leakage a random/shuffled K-fold split would introduce.

Metrics (same as the baseline and random forest notebooks, for direct comparison):
- **MAE** (Mean Absolute Error) — average position error
- **Spearman ρ** — rank-order correlation between predicted and actual finishing positions
- **Macro F1** — treats each position (1–20) as a class; averages F1 equally across all positions regardless of frequency

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, f1_score
from scipy.stats import spearmanr

sys.path.insert(0, str(Path.cwd().parent.parent))
from pipeline.feature_engineering.feature_engineering import FINAL_FEATURES, TARGET

## 2. Load Data

Train/test splits were produced by the feature engineering pipeline and persisted as Parquet files under `data/`. `train.parquet` holds seasons 2019–2024; `test.parquet` holds the held-out 2025 season.

In [2]:
train = pd.read_parquet('../../data/train.parquet')
test = pd.read_parquet('../../data/test.parquet')

print(f"train: {train.shape[0]} rows, years {sorted(train['year'].unique())}")
print(f"test:  {test.shape[0]} rows, years {sorted(test['year'].unique())}")

train: 2556 rows, years [np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]
test:  479 rows, years [np.int32(2025)]


## 3. Prepare Features and Target

`RacePosition` (1–20) is the regression target. Model inputs are restricted to `FINAL_FEATURES` — the locked feature list from the feature engineering pipeline — which excludes identifier columns (`DriverId`, `DriverNumber`, `year`) that carry no predictive signal on their own.

In [3]:
train_X = train[FINAL_FEATURES]
train_Y = train[TARGET]
test_X = test[FINAL_FEATURES]
test_Y = test[TARGET]

## 4. Validate Data

Confirm no missing values before training — the feature engineering pipeline should have handled imputation upstream.

In [4]:
for name, df in [("train_X", train_X), ("train_Y", train_Y), ("test_X", test_X), ("test_Y", test_Y)]:
    n_null = df.isna().sum().sum() if hasattr(df, "columns") else df.isna().sum()
    status = "OK" if n_null == 0 else f"WARNING: {n_null} nulls"
    print(f"{name}: {status}")

train_X: OK
train_Y: OK
test_X: OK
test_Y: OK


## 5. Metric Helpers

Same definitions as the baseline and random forest notebooks, so scores are directly comparable.

In [5]:
def macro_f1(y_true, y_pred):
    """Round continuous predictions to nearest integer position, then compute macro F1."""
    y_pred_int = pd.Series(y_pred).round().clip(1, 20).astype(int)
    return f1_score(y_true.astype(int), y_pred_int, average="macro", zero_division=0)

def safe_spearmanr(y_true, y_pred):
    """Return 0.0 when predictions are constant (ρ is undefined for constant input)."""
    if pd.Series(y_pred).nunique() == 1:
        return 0.0
    return spearmanr(y_true, y_pred).statistic

def score(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "Spearman_rho": safe_spearmanr(y_true, y_pred),
        "Macro_F1": macro_f1(y_true, y_pred),
    }

## 6. Temporal Cross-Validation

Expanding-window walk-forward split over the 6 training seasons (2019–2024): fold *i* trains on every season strictly before the validation season and validates on the next one. This yields 5 folds:

| Fold | Train years | Validation year |
|---|---|---|
| 1 | 2019 | 2020 |
| 2 | 2019–2020 | 2021 |
| 3 | 2019–2021 | 2022 |
| 4 | 2019–2022 | 2023 |
| 5 | 2019–2023 | 2024 |

`LGBMRegressor` is trained with default hyperparameters (only `random_state` is fixed, for reproducibility, and `verbose=-1` to silence LightGBM's training logs) in every fold.

In [6]:
years = sorted(train['year'].unique())
cv_rows = []

for i in range(1, len(years)):
    train_years = years[:i]
    val_year = years[i]

    fold_train_mask = train['year'].isin(train_years)
    fold_val_mask = train['year'] == val_year

    fold_model = LGBMRegressor(random_state=42, verbose=-1)
    fold_model.fit(train_X[fold_train_mask], train_Y[fold_train_mask])
    fold_preds = fold_model.predict(train_X[fold_val_mask])

    metrics = score(train_Y[fold_val_mask], fold_preds)
    metrics["fold"] = i
    metrics["train_years"] = f"{train_years[0]}-{train_years[-1]}" if len(train_years) > 1 else str(train_years[0])
    metrics["val_year"] = val_year
    cv_rows.append(metrics)

cv_results = pd.DataFrame(cv_rows)[["fold", "train_years", "val_year", "MAE", "Spearman_rho", "Macro_F1"]]
cv_results

,fold,train_years,val_year,MAE,Spearman_rho,Macro_F1
0,1,2019,2020,4.018426,0.506021,0.083806
1,2,2019-2020,2021,3.748084,0.568852,0.090464
2,3,2019-2021,2022,3.783869,0.540800,0.069471
3,4,2019-2022,2023,3.436395,0.613626,0.116530
4,5,2019-2023,2024,3.225506,0.694280,0.094158


## 7. CV Summary

In [7]:
cv_summary = cv_results[["MAE", "Spearman_rho", "Macro_F1"]].agg(["mean", "std"])
print(f"{'Metric':<14} {'Mean':>8}  {'Std':>8}")
print("-" * 34)
for col in ["MAE", "Spearman_rho", "Macro_F1"]:
    print(f"{col:<14} {cv_summary.loc['mean', col]:>8.3f}  {cv_summary.loc['std', col]:>8.3f}")

Metric             Mean       Std
----------------------------------
MAE               3.642     0.312
Spearman_rho      0.585     0.073
Macro_F1          0.091     0.017


## 8. Final Model — Evaluate on Held-Out 2025 Test Set

Refit on the full training set (2019–2024) and score once on the untouched 2025 test season, alongside the baselines and Random Forest results already saved in `reports/random_forest_test_results.csv` for comparison.

In [8]:
final_model = LGBMRegressor(random_state=42, verbose=-1)
final_model.fit(train_X, train_Y)
test_preds = final_model.predict(test_X)

test_results = score(test_Y, test_preds)

prior_results = pd.read_csv("../../reports/random_forest_test_results.csv").set_index("model")
comparison = pd.concat([
    prior_results,
    pd.DataFrame({"LightGBM": test_results}).T.rename_axis("model"),
])

print(f"{'Model':<20} {'MAE':>6}  {'Spearman ρ':>10}  {'Macro F1':>8}")
print("-" * 52)
for model, metrics in comparison.iterrows():
    print(f"{model:<20} {metrics['MAE']:>6.2f}  {metrics['Spearman_rho']:>10.3f}  {metrics['Macro_F1']:>8.3f}")

Model                   MAE  Spearman ρ  Macro F1
----------------------------------------------------
GridPosition          10.44       0.652     0.005
DummyRegressor         4.99       0.000     0.005
RandomForest           3.52       0.614     0.080
LightGBM               3.52       0.608     0.088


## 9. Train vs CV Error — Overfitting Check

Compare error on the data `final_model` was fit on (train, in-sample) against the mean CV error (out-of-sample, Section 7) and the held-out 2025 test error (Section 8). A large train-vs-CV/test gap signals overfitting — a real risk here since the model above uses unrestricted depth / unlimited boosting rounds by default.

In [9]:
train_preds = final_model.predict(train_X)
train_results = score(train_Y, train_preds)

error_comparison = pd.DataFrame({
    "Train (in-sample)": train_results,
    "CV (mean, out-of-sample)": cv_summary.loc["mean"],
    "Test (2025, held-out)": test_results,
}).T[["MAE", "Spearman_rho", "Macro_F1"]]

error_comparison

,MAE,Spearman_rho,Macro_F1
Train (in-sample),2.107542,0.886063,0.158878
"CV (mean, out-of-sample)",3.642456,0.584716,0.090886
"Test (2025, held-out)",3.520125,0.608012,0.087795


## 10. Feature Importance

First-pass feature importance from `final_model` (fit on the full 2019–2024 training set in Section 8). The scikit-learn API's default `importance_type='split'` counts how often a feature is used to split, not its impact on error — a different basis than Random Forest's impurity-based importance, so magnitudes aren't directly comparable across the two models, only the relative ranking within each.

In [10]:
feature_importance = pd.Series(
    final_model.feature_importances_, index=FINAL_FEATURES, name="importance"
).sort_values(ascending=False)

feature_importance

DriverFinish_ewm               471
TeamFinish_ewm                 455
LapStd_lag1                    447
TeamFinish_roll3_inseason      338
DriverFinish_roll3_inseason    270
GridPosition                   248
Meeting.Circuit.ShortName      211
round_number                   201
DriverFinish_lag1              197
TeamName                       162
Name: importance, dtype: int32

## 10a. Multicollinearity Check (VIF)

Variance Inflation Factor (VIF) measures how much a feature's variance is inflated by linear correlation with the other features — VIF > 5 is the usual "moderately collinear" threshold. Computed on the 8 continuous/rolling features only; `TeamName` and `Meeting.Circuit.ShortName` are label-encoded categoricals with an arbitrary numeric ordering, so a linear-redundancy statistic isn't meaningful for them — they're excluded here and kept regardless of VIF. (Same computation and result as the Random Forest notebook, Section 10a — VIF is a property of the feature matrix, not the model.)

In [11]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

CATEGORICAL_FEATURES = ["TeamName", "Meeting.Circuit.ShortName"]
vif_features = [f for f in FINAL_FEATURES if f not in CATEGORICAL_FEATURES]

vif_X = add_constant(train_X[vif_features].dropna())
vif = pd.Series(
    [variance_inflation_factor(vif_X.values, i) for i in range(vif_X.shape[1])],
    index=vif_X.columns,
    name="VIF",
).drop("const").sort_values(ascending=False)

vif

TeamFinish_ewm                 25.175282
DriverFinish_ewm               25.139299
TeamFinish_roll3_inseason      22.441007
DriverFinish_roll3_inseason    18.338941
DriverFinish_lag1               3.513101
GridPosition                    1.711442
round_number                    1.002871
LapStd_lag1                     1.001978
Name: VIF, dtype: float64

## 10b. Feature Selection — Drop Near-Zero-Importance, High-VIF Features

Two independent signals both have to flag a feature before it's dropped:
- **Near-zero importance** — split-count importance share below `1 / 10` features (10%), i.e. contributing less than a uniformly-weighted feature would.
- **High VIF** — VIF > 5 (Section 10a).

A feature that's collinear but still pulling its weight (e.g. `TeamFinish_ewm` / `DriverFinish_ewm` — both high VIF *and* top-3 importance) is kept; only the redundant, low-importance side of a collinear pair gets cut. Unlike Random Forest, LightGBM spreads more importance onto `TeamFinish_roll3_inseason`, so it survives the near-zero-importance bar here even though it's high-VIF.

In [12]:
IMPORTANCE_THRESHOLD = 1 / len(FINAL_FEATURES)  # 10% — a uniformly-weighted feature's share
VIF_THRESHOLD = 5

importance_share = feature_importance / feature_importance.sum()

selection = pd.DataFrame({
    "importance_share": importance_share,
    "VIF": vif.reindex(FINAL_FEATURES),
})
selection["near_zero_importance"] = selection["importance_share"] < IMPORTANCE_THRESHOLD
selection["high_vif"] = selection["VIF"] > VIF_THRESHOLD
selection["drop"] = selection["near_zero_importance"] & selection["high_vif"].fillna(False)

SELECTED_FEATURES = selection[~selection["drop"]].index.tolist()
dropped_features = selection[selection["drop"]].index.tolist()

print(f"Dropped ({len(dropped_features)}): {dropped_features}")
print(f"Retained ({len(SELECTED_FEATURES)}): {SELECTED_FEATURES}\n")

selection.sort_values("importance_share", ascending=False)

Dropped (1): ['DriverFinish_roll3_inseason']
Retained (9): ['DriverFinish_ewm', 'DriverFinish_lag1', 'GridPosition', 'LapStd_lag1', 'Meeting.Circuit.ShortName', 'TeamFinish_ewm', 'TeamFinish_roll3_inseason', 'TeamName', 'round_number']



,importance_share,VIF,near_zero_importance,high_vif,drop
DriverFinish_ewm,0.157000,25.139299,False,True,False
TeamFinish_ewm,0.151667,25.175282,False,True,False
LapStd_lag1,0.149000,1.001978,False,False,False
TeamFinish_roll3_inseason,0.112667,22.441007,False,True,False
DriverFinish_roll3_inseason,0.090000,18.338941,True,True,True
GridPosition,0.082667,1.711442,True,False,False
Meeting.Circuit.ShortName,0.070333,NaN,True,False,False
round_number,0.067000,1.002871,True,False,False
DriverFinish_lag1,0.065667,3.513101,True,False,False
TeamName,0.054000,NaN,True,False,False


## 10c. Refit and Compare with Reduced Feature Set

Rerun the same temporal CV (Section 6) and held-out 2025 evaluation (Section 8), restricted to `SELECTED_FEATURES`, and compare against the all-features model above.

In [13]:
def run_cv(feature_cols):
    rows = []
    for i in range(1, len(years)):
        cv_train_years = years[:i]
        val_year = years[i]

        fold_train_mask = train['year'].isin(cv_train_years)
        fold_val_mask = train['year'] == val_year

        fold_model = LGBMRegressor(random_state=42, verbose=-1)
        fold_model.fit(train[feature_cols][fold_train_mask], train_Y[fold_train_mask])
        fold_preds = fold_model.predict(train[feature_cols][fold_val_mask])

        metrics = score(train_Y[fold_val_mask], fold_preds)
        metrics["fold"] = i
        metrics["train_years"] = f"{cv_train_years[0]}-{cv_train_years[-1]}" if len(cv_train_years) > 1 else str(cv_train_years[0])
        metrics["val_year"] = val_year
        rows.append(metrics)
    return pd.DataFrame(rows)[["fold", "train_years", "val_year", "MAE", "Spearman_rho", "Macro_F1"]]

reduced_cv_results = run_cv(SELECTED_FEATURES)
reduced_cv_summary = reduced_cv_results[["MAE", "Spearman_rho", "Macro_F1"]].agg(["mean", "std"])

reduced_model = LGBMRegressor(random_state=42, verbose=-1)
reduced_model.fit(train[SELECTED_FEATURES], train_Y)
reduced_test_results = score(test_Y, reduced_model.predict(test[SELECTED_FEATURES]))

feature_selection_comparison = pd.DataFrame({
    "All features (CV mean)": cv_summary.loc["mean"],
    "Reduced features (CV mean)": reduced_cv_summary.loc["mean"],
    "All features (2025 test)": pd.Series(test_results),
    "Reduced features (2025 test)": pd.Series(reduced_test_results),
}).T[["MAE", "Spearman_rho", "Macro_F1"]]

feature_selection_comparison

,MAE,Spearman_rho,Macro_F1
All features (CV mean),3.642456,0.584716,0.090886
Reduced features (CV mean),3.635145,0.586482,0.083988
All features (2025 test),3.520125,0.608012,0.087795
Reduced features (2025 test),3.542820,0.598368,0.092017


## 11. Save Results

Persists the reduced-feature-set CV results and test comparison (Section 10c) — the model actually selected after the VIF/importance check — not the diagnostic all-10-feature run from Sections 6–8.

In [14]:
final_comparison = pd.concat([
    prior_results,
    pd.DataFrame({"LightGBM": reduced_test_results}).T.rename_axis("model"),
])

reduced_cv_results.to_csv("../../reports/lightgbm_cv_results.csv", index=False)
final_comparison.reset_index().to_csv("../../reports/lightgbm_test_results.csv", index=False)
final_comparison

,MAE,Spearman_rho,Macro_F1
model,,,
GridPosition,10.437773,0.651649,0.005263
DummyRegressor,4.990574,0.000000,0.004771
RandomForest,3.515595,0.614057,0.079802
LightGBM,3.542820,0.598368,0.092017


## 11a. Optuna-Tuned Hyperparameters — Refit and Compare

Loads the winning hyperparameters from `reports/lightgbm_best_params.json`, produced by `pipeline/model/tune_lightgbm.py` — a 50-trial Optuna search (TPE sampler, `MedianPruner`) minimizing mean MAE across the same 5-fold expanding-window temporal CV as Section 6. The tuning script only ever reads `train.parquet`, so the 2025 test season stayed fully unseen throughout the search, and trials were selected by CV score, never train-set error.

The search optimized against `FINAL_FEATURES` (all 10 features), not the `SELECTED_FEATURES` reduced set from Section 10c — refitting here on that same 10-feature matrix so the reported CV/test scores reflect what was actually searched.

In [15]:
import json

with open("../../reports/lightgbm_best_params.json") as fh:
    tuning_result = json.load(fh)

tuned_params = tuning_result["params"]
print(f"Optuna CV MAE: {tuning_result['cv_mae']:.4f}")
print(f"Tuned params: {tuned_params}")

Optuna CV MAE: 3.3969
Tuned params: {'n_estimators': 614, 'num_leaves': 40, 'max_depth': 3, 'learning_rate': 0.008381172712733035, 'min_child_samples': 17, 'reg_alpha': 0.4561618416669044, 'reg_lambda': 0.017167180659618215, 'subsample': 0.8307963751746064, 'colsample_bytree': 0.7133774016145881}


In [16]:
def run_cv_with_params(feature_cols, params):
    rows = []
    for i in range(1, len(years)):
        cv_train_years = years[:i]
        val_year = years[i]

        fold_train_mask = train['year'].isin(cv_train_years)
        fold_val_mask = train['year'] == val_year

        fold_model = LGBMRegressor(random_state=42, verbose=-1, **params)
        fold_model.fit(train[feature_cols][fold_train_mask], train_Y[fold_train_mask])
        fold_preds = fold_model.predict(train[feature_cols][fold_val_mask])

        metrics = score(train_Y[fold_val_mask], fold_preds)
        metrics["fold"] = i
        metrics["train_years"] = f"{cv_train_years[0]}-{cv_train_years[-1]}" if len(cv_train_years) > 1 else str(cv_train_years[0])
        metrics["val_year"] = val_year
        rows.append(metrics)
    return pd.DataFrame(rows)[["fold", "train_years", "val_year", "MAE", "Spearman_rho", "Macro_F1"]]

tuned_cv_results = run_cv_with_params(FINAL_FEATURES, tuned_params)
tuned_cv_summary = tuned_cv_results[["MAE", "Spearman_rho", "Macro_F1"]].agg(["mean", "std"])

tuned_model = LGBMRegressor(random_state=42, verbose=-1, **tuned_params)
tuned_model.fit(train_X, train_Y)
tuned_test_results = score(test_Y, tuned_model.predict(test_X))

tuned_cv_summary

,MAE,Spearman_rho,Macro_F1
mean,3.396944,0.645180,0.068595
std,0.279094,0.067755,0.011218


In [17]:
tuning_comparison = pd.DataFrame({
    "Default params, all features (CV mean)": cv_summary.loc["mean"],
    "Reduced features (CV mean)": reduced_cv_summary.loc["mean"],
    "Optuna-tuned, all features (CV mean)": tuned_cv_summary.loc["mean"],
    "Default params, all features (2025 test)": pd.Series(test_results),
    "Reduced features (2025 test)": pd.Series(reduced_test_results),
    "Optuna-tuned, all features (2025 test)": pd.Series(tuned_test_results),
}).T[["MAE", "Spearman_rho", "Macro_F1"]]

tuning_comparison

,MAE,Spearman_rho,Macro_F1
"Default params, all features (CV mean)",3.642456,0.584716,0.090886
Reduced features (CV mean),3.635145,0.586482,0.083988
"Optuna-tuned, all features (CV mean)",3.396944,0.645180,0.068595
"Default params, all features (2025 test)",3.520125,0.608012,0.087795
Reduced features (2025 test),3.542820,0.598368,0.092017
"Optuna-tuned, all features (2025 test)",3.344710,0.651619,0.057819


In [18]:
tuned_cv_results.to_csv("../../reports/lightgbm_tuned_cv_results.csv", index=False)

tuned_comparison = pd.concat([
    prior_results,
    pd.DataFrame({"LightGBM (Optuna-tuned)": tuned_test_results}).T.rename_axis("model"),
])
tuned_comparison.reset_index().to_csv("../../reports/lightgbm_tuned_test_results.csv", index=False)
tuned_comparison

,MAE,Spearman_rho,Macro_F1
model,,,
GridPosition,10.437773,0.651649,0.005263
DummyRegressor,4.990574,0.000000,0.004771
RandomForest,3.515595,0.614057,0.079802
LightGBM (Optuna-tuned),3.344710,0.651619,0.057819


## 12. Takeaways

- The temporal CV folds show how default-parameter LightGBM performance evolves as more historical seasons become available for training — early folds (fold 1: train on a single season) are a much weaker test than later folds.
- The final row compares the model, fit on all six training seasons, against the `GridPosition`/`DummyRegressor` baselines and `RandomForest` on the untouched 2025 season.
- **Overfitting is present but narrower than Random Forest**: train MAE is 2.11 vs. 3.64 on CV and 3.52 on test — a ~1.5-point gap, versus RandomForest's ~2.3-point gap (see its notebook, Section 9). LightGBM's boosted, shallower trees generalize somewhat better out of the box, though the gap is still large enough that `num_leaves`, `min_child_samples`, and early stopping on `n_estimators` are worth tuning.
- **Feature importance (split-count) is more evenly spread**: on the diagnostic all-10-feature fit, `DriverFinish_ewm`, `TeamFinish_ewm`, and `LapStd_lag1` lead, while `GridPosition` ranks 6th of 10 — a notable divergence from Random Forest, where `GridPosition` is the single dominant feature. LightGBM's boosting spreads splits across the engineered rolling/EWM form features rather than concentrating on qualifying position.
- **Multicollinearity check drops 1 redundant feature (Sections 10a–10c)**: the same VIF pass as Random Forest flags 4 features as highly collinear (VIF 18–25), but LightGBM's importance is spread more evenly, so only `DriverFinish_roll3_inseason` (9.0% share, below the 10% near-zero bar) meets both the near-zero-importance and high-VIF criteria — `TeamFinish_roll3_inseason` (11.3% share) survives here even though Random Forest drops it too. Refitting on the remaining 9 `SELECTED_FEATURES` leaves metrics within CV noise (CV MAE 3.642→3.635, test MAE 3.520→3.543; CV std is 0.312). `lightgbm_cv_results.csv` / `lightgbm_test_results.csv` now reflect this reduced 9-feature model.
- **Optuna tuning (Section 11a) improves MAE and rank correlation but trades away Macro F1**: a 50-trial temporal-CV search (`pipeline/model/tune_lightgbm.py`) lands on a much slower, shallower model (`learning_rate` 0.0084, `max_depth` 3, `n_estimators` 614) than the untuned defaults, cutting test MAE from 3.52 (all-features) to 3.34 and lifting Spearman ρ from 0.61 to 0.65 — but test Macro F1 drops from ~0.09 to 0.058. The tuning objective was MAE only, so the search had no incentive to keep predictions spread across all 20 position classes; the lower learning rate likely produces smoother, more conservative predictions that round to a narrower band of positions, which MAE/rank correlation reward but per-class F1 penalizes. Whether that tradeoff is acceptable depends on whether downstream use cares more about "how far off" (MAE) or "exact position hit rate" (Macro F1). Tuned results are saved separately to `lightgbm_tuned_cv_results.csv` / `lightgbm_tuned_test_results.csv` — the "official" `lightgbm_cv_results.csv` / `lightgbm_test_results.csv` still reflect the untuned, reduced-feature model referenced elsewhere (e.g. `shap_analysis.ipynb`, README).